# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore a dataset defined via a Croissant schema using the `mlcroissant` library.

### Dataset Source
The dataset source is provided as a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display name and description
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")

## 2. Data Overview
Review available record sets and their field `@id`s. All references use the Croissant schema's `@id` fields for reproducibility and clarity.

In [ ]:
# List available record sets and their fields
print("Available Record Sets (referenced by @id):\n")
for rs in dataset.metadata.record_sets:
    print(f"Record Set: {rs['@id']}")
    print(f"  Name: {rs.get('name','')}")
    print(f"  Description: {rs.get('description','')}")
    print("  Fields:")
    for field in rs['field']:
        field_obj = dataset.metadata.get_entity(field['@id'])
        print(f"    - @id: {field_obj['@id']} / name: {field_obj.get('name', '')} / dataType: {field_obj.get('dataType', '')}")
    print()

## 3. Data Extraction
Load data from a specific record set (`@id`) into a DataFrame for analysis. All record set and field IDs are referenced by their Croissant `@id`.

In [ ]:
# Use the available record sets identified above
# (You may want to adjust this list according to the output of the previous cell)
record_sets = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded record set {record_set_id} with {len(records)} records. Columns:")
        print(dataframes[record_set_id].columns.tolist())
        print()
    except Exception as e:
        print(f"Could not load records for record set {record_set_id}: {e}\n")

# Show the head of the first record set loaded (based on the order in record_sets)
main_record_set = record_sets[0]
dataframes[main_record_set].head()

## 4. Exploratory Data Analysis (EDA)
Let's process the main table. We'll select a numeric field (e.g., coverage percentage, peptide counts, or molecular weight), filter, normalize, and group. **All field selections use their `@id`.**

In [ ]:
# Replace these IDs based on the output from Section 2 and 3
# As an example, suppose these field @id s are present:
#   - 'ev:coverage_percent' (coverage%)
#   - 'ev:mw' (molecular weight)
#   - 'ev:sample_name' (sample/class/category)
#   - 'ev:unique_peptides' (number of unique peptides)

# Find a numeric field (replace with actual field @id available in the dataset)
candidate_numeric_ids = [col for col in dataframes[main_record_set].columns if 'coverage' in col or 'mw' in col or 'peptide' in col or 'abundance' in col or 'count' in col or 'number' in col]
print("Numeric field candidates:", candidate_numeric_ids)

# Pick the first candidate as numeric_field
numeric_field = candidate_numeric_ids[0] if candidate_numeric_ids else dataframes[main_record_set].columns[0]
group_field = None
# Try to find a grouping field
for col in dataframes[main_record_set].columns:
    if 'sample' in col or 'class' in col or 'type' in col or 'category' in col:
        group_field = col
        break

print(f"Selected numeric field for EDA: {numeric_field}")
if group_field:
    print(f"Grouping field: {group_field}")

# Convert column to numeric (if needed)
df = dataframes[main_record_set].copy()
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
threshold = df[numeric_field].mean() if df[numeric_field].notnull().any() else 0

# Filter rows above the threshold
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:0.3f} (mean): {len(filtered_df)} rows")
print(filtered_df.head())

# Normalize numeric_field (z-score)
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a field if available
if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(numeric_field, ascending=False)
    print(f"Grouped data mean {numeric_field} by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or field relationships. Here, we plot the distribution of the selected numeric field, and if possible, compare across groups.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of numeric field
plt.figure(figsize=(6,4))
sns.histplot(df[numeric_field].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If grouping field is available, boxplot by group
if group_field and group_field in df.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=df[group_field], y=df[numeric_field])
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to:

- Load and inspect dataset metadata and records via the Croissant schema using `mlcroissant`.
- Identify and extract data from record sets and fields using their `@id` for reproducibility.
- Apply exploratory data analysis (EDA) operations on numeric fields and visualize the findings.

This approach supports fast and FAIR-compliant (Findable, Accessible, Interoperable, Reusable) dataset exploration and lays the foundation for downstream ML or scientific workflows.